# Entrega 3 · Machine Learning: estimar la proteína de los productos sin datos

En la entrega 2 preparé los datos de Mercadona y de Open Food Facts hasta tener un Gold con los nutrientes de cada producto. El problema es que algunos productos no aparecen en Open Food Facts o tienen los datos incompletos, y sin nutrientes mi producto no puede recomendarlos.

En este notebook pruebo si un modelo de Machine Learning puede **estimar los gramos de proteína por 100 g** de esos productos usando solo lo que Mercadona sí publica (nombre, categoría, precio y formato). Es un problema de regresión.

**Qué hago, en orden:**
1. Cargo el Gold ampliado y reviso los datos.
2. Preparo las variables y separo entrenamiento y test.
3. Comparo un baseline sencillo con dos modelos (primera ronda).
4. Pruebo una variante que mejora la idea (segunda ronda).
5. Compruebo si la mejora es real, qué pasaría sin el control de fugas y qué rango de error tiene la estimación.
6. Fijo el modelo elegido y genero la salida que usará el producto.

**Para reproducirlo:** primero hay que ejecutar `pipeline_ampliado.ipynb`, que genera `data/gold/productos_gold_ampliado.csv`. Los datos no se publican en el repositorio; las métricas sí están en `resultados/`.

## 1. Cargar el Gold ampliado y mirar los datos

Cargo el Gold ampliado, que es el de la entrega 2 llevado de 3 a 43 subcategorías de alimentos envasados (1037 productos en lugar de 70). Con 70 productos un modelo no se puede validar de forma seria.

Antes de entrenar nada compruebo tres cosas:
- **Qué columnas tienen huecos** entre las que usaría como entrada.
- **Qué productos tienen la proteína más alta**, por si hay errores de origen.
- **Si hay productos con proteína 0** en el grupo base, que es raro en alimentos como queso o embutido.

In [1]:
import pandas as pd

gold = pd.read_csv("data/gold/productos_gold_ampliado.csv", dtype={"ean": str})
print(gold.shape)

print("\nNulos en las columnas que usaría el modelo como entrada:")
entradas = ["nombre", "categoria_id", "grupo", "formato", "estado",
            "precio_unidad", "precio_kg", "tamano_envase"]
print(gold[entradas].isna().sum())

print("\nProteína más alta (posibles errores o productos especiales):")
cols = ["nombre", "grupo", "proteina_100g", "kcal_100g", "estado_calidad"]
print(gold.sort_values("proteina_100g", ascending=False)[cols].head(15).to_string())

print("\nProteína 0 en el grupo base:")
print(gold[(gold["grupo"] == "base") & (gold["proteina_100g"] == 0)][cols].head(15).to_string())

(1037, 22)

Nulos en las columnas que usaría el modelo como entrada:
nombre             0
categoria_id       0
grupo              0
formato          114
estado             0
precio_unidad      0
precio_kg          7
tamano_envase      0
dtype: int64

Proteína más alta (posibles errores o productos especiales):
                                                                     nombre        grupo  proteina_100g   kcal_100g  estado_calidad
296                Torreznillos Hacendado cortezas de cerdo fritas ahumadas  snack_dulce           59.0  570.000000       coherente
283                                                     Torreznos Hacendado  snack_dulce           52.0  600.000000       coherente
23                                                  Soja texturizada Luengo         base           50.0  361.000000       coherente
237     Cacahuete en polvo desgrasado +Proteínas 14 g 70% reducido en grasa         base           46.7  416.666667       coherente
281                       Co

## 2. Preparar los datos

**Qué vi en el paso anterior:**
- `formato` tiene 114 huecos. Los relleno con "desconocido" en vez de descartar esos productos.
- Las proteínas más altas (torreznos, cortezas, soja texturizada, jamón) son valores reales y los dejo.
- Un producto sí es un error: **"Queso curado mezcla con trufa Hacendado"** aparece con 0 g de proteína y 900 kcal, es decir, grasa pura. Un queso no es así. Es un error de Open Food Facts que el control de calorías de la entrega 2 no detecta, porque la comprobación 4/4/9 cuadra con esos números. Lo excluyo del entrenamiento y lo dejo aquí escrito.

**Qué variables uso:** solo lo que Mercadona publica antes de conocer los nutrientes: categoría, nombre, precios, tamaño, formato, estado y si es Hacendado. Para el precio por kilo uso `precio_kg`; los pocos productos cuyo precio no es por kilo se rellenan con la mediana. **No uso calorías, grasa ni hidratos como entrada**, porque un producto sin datos en Open Food Facts tampoco los tiene, y además estarían muy ligados a la proteína (sería una fuga de información).

**Variante del producto.** Creo una columna con las dos primeras palabras del nombre sin la marca (por ejemplo, "pipas girasol"). La uso más adelante para que productos casi iguales no queden uno en entrenamiento y otro en test.

In [2]:
import numpy as np

# 1) Exclusión documentada: el queso con trufa tiene 0 g de proteína y 900 kcal,
#    que es un error de Open Food Facts (un queso no es grasa pura)
mask_error = (gold["nombre"] == "Queso curado mezcla con trufa Hacendado") & (gold["proteina_100g"] == 0)
print("Productos excluidos del entrenamiento por error de origen:", mask_error.sum())
print(gold.loc[mask_error, ["ean", "nombre", "kcal_100g", "proteina_100g", "grasa_100g", "hidratos_100g"]].to_string())
datos = gold[~mask_error].copy()

# 2) Variables de entrada: solo lo que Mercadona publica ANTES de conocer los nutrientes
datos["categoria_id"] = datos["categoria_id"].astype(str)
datos["formato"] = datos["formato"].fillna("desconocido")
datos["es_hacendado"] = datos["nombre"].str.contains("hacendado", case=False).astype(int)
datos["log_precio_ref"] = np.log1p(datos["precio_kg"].fillna(datos["precio_kg"].median()))
datos["log_precio_ud"] = np.log1p(datos["precio_unidad"])
datos["log_tamano"] = np.log1p(datos["tamano_envase"])

# 3) Grupo de variante: las dos primeras palabras del nombre sin la marca.
#    Sirve para que variantes casi iguales no queden una en train y otra en test (fuga de información)
datos["variante"] = (datos["nombre"].str.lower().str.replace("hacendado", "", regex=False)
                     .str.split().str[:2].str.join(" "))
print("\nProductos:", len(datos), "| variantes distintas:", datos["variante"].nunique())
print(datos["variante"].value_counts().head(8))

Productos excluidos del entrenamiento por error de origen: 1
               ean                                   nombre  kcal_100g  proteina_100g  grasa_100g  hidratos_100g
501  8480000048837  Queso curado mezcla con trufa Hacendado      900.0            0.0       100.0            0.0

Productos: 1036 | variantes distintas: 580
variante
pan de                21
barritas de           18
salchichas cocidas    14
queso fresco          13
patatas fritas        11
chocolate con         10
postre lácteo         10
chocolate negro       10
Name: count, dtype: int64


## 3. Primera ronda: baseline y dos modelos

**Separación de datos.** 80 % para entrenar y 20 % para test, agrupando por variante. Así el modelo no ve en test una variante casi idéntica a la que vio entrenando.

**Baseline.** Predigo, para cada producto, la proteína media de su categoría (calculada solo con el entrenamiento). Es lo que haría una persona sin ML: un yogur no tiene la proteína de un jamón. Cualquier modelo tiene que ganarle para justificar su complejidad.

**Modelos de ML:**
- **Ridge:** regresión lineal con penalización, buena con pocos datos y muchas columnas de texto.
- **Random Forest:** un modelo no lineal, por si hay relaciones que Ridge no capta.

**Cómo comparo.** Con validación cruzada de 5 pliegues, también agrupada por variante, **solo sobre el entrenamiento**. El test queda para el final.

**Métricas.** El **MAE** es el error medio en gramos por 100 g (la más fácil de entender). El RMSE penaliza más los errores grandes. El R² dice qué parte de la variación explica el modelo. `pct_error_<2g` es el porcentaje de productos con error menor de 2 g.

**Qué mirar en la salida:** si algún modelo baja el MAE del baseline y si la validación cruzada y el test cuentan la misma historia.

In [3]:
import os
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

OBJETIVO = "proteina_100g"

# --- Separación: 80 % entrenamiento / 20 % test, sin mezclar variantes entre ambos ---
idx_tr, idx_te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
                      .split(datos, groups=datos["variante"]))
train, test = datos.iloc[idx_tr], datos.iloc[idx_te]
print("Entrenamiento:", len(train), "| Test:", len(test))

def metricas(real, pred):
    err = pred - real
    return {"MAE": np.abs(err).mean(),
            "RMSE": np.sqrt((err ** 2).mean()),
            "R2": 1 - (err ** 2).sum() / ((real - real.mean()) ** 2).sum(),
            "pct_error_<2g": (np.abs(err) < 2).mean() * 100}

# --- Baseline: proteína media de la categoría (con la media global si la categoría no existe) ---
def predecir_baseline(entrenamiento, df):
    media_cat = entrenamiento.groupby("categoria_id")[OBJETIVO].mean()
    return df["categoria_id"].map(media_cat).fillna(entrenamiento[OBJETIVO].mean()).values

# --- Modelos de ML ---
def preprocesado():
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["categoria_id", "estado", "formato"]),
        ("txt", TfidfVectorizer(ngram_range=(1, 2), min_df=2, strip_accents="unicode"), "nombre"),
        ("num", StandardScaler(), ["es_hacendado", "log_precio_ref", "log_precio_ud", "log_tamano"]),
    ])

modelos = {
    "Ridge": lambda: Pipeline([("pre", preprocesado()), ("m", Ridge(alpha=1.0))]),
    "Random Forest": lambda: Pipeline([("pre", preprocesado()),
                                       ("m", RandomForestRegressor(n_estimators=300, min_samples_leaf=2,
                                                                   n_jobs=-1, random_state=42))]),
}

# --- Validación cruzada agrupada SOLO sobre entrenamiento (para comparar modelos) ---
cv = GroupKFold(n_splits=5)
cv_mae = {"Baseline (media por categoría)": []}
cv_mae.update({n: [] for n in modelos})
for a, b in cv.split(train, groups=train["variante"]):
    tr_f, va_f = train.iloc[a], train.iloc[b]
    cv_mae["Baseline (media por categoría)"].append(
        metricas(va_f[OBJETIVO].values, predecir_baseline(tr_f, va_f))["MAE"])
    for n, f in modelos.items():
        m = f().fit(tr_f, tr_f[OBJETIVO])
        cv_mae[n].append(metricas(va_f[OBJETIVO].values, m.predict(va_f))["MAE"])

# --- Evaluación final en test (datos que el modelo no ha visto) ---
pred_test = {"Baseline (media por categoría)": predecir_baseline(train, test)}
for n, f in modelos.items():
    pred_test[n] = f().fit(train, train[OBJETIVO]).predict(test)

filas = []
for n, p in pred_test.items():
    r = metricas(test[OBJETIVO].values, p)
    r.update({"modelo": n, "MAE_cv_media": np.mean(cv_mae[n]), "MAE_cv_desv": np.std(cv_mae[n])})
    filas.append(r)
resultados = pd.DataFrame(filas).set_index("modelo").round(2)
print(resultados.to_string())

os.makedirs("resultados", exist_ok=True)
resultados.to_csv("resultados/metricas_ml.csv")

# --- Análisis de errores del mejor modelo según validación cruzada ---
mejor = min(modelos, key=lambda n: np.mean(cv_mae[n]))
print("\nMejor modelo por validación cruzada:", mejor)
t = test.copy()
t["pred"] = pred_test[mejor]
t["pred_baseline"] = pred_test["Baseline (media por categoría)"]
t["error_abs"] = (t["pred"] - t[OBJETIVO]).abs()
t["error_abs_baseline"] = (t["pred_baseline"] - t[OBJETIVO]).abs()

print("\nError medio (g) por grupo, modelo vs baseline:")
print(t.groupby("grupo")[["error_abs", "error_abs_baseline"]].mean().round(2))
print("\nError medio (g) Hacendado (1) vs otras marcas (0):")
print(t.groupby("es_hacendado")[["error_abs", "error_abs_baseline"]].agg(["mean", "count"]).round(2))
print("\n10 mayores errores:")
print(t.sort_values("error_abs", ascending=False)[["nombre", "grupo", OBJETIVO, "pred"]].head(10).round(1).to_string())

Entrenamiento: 849 | Test: 187
                                 MAE  RMSE    R2  pct_error_<2g  MAE_cv_media  MAE_cv_desv
modelo                                                                                    
Baseline (media por categoría)  3.39  5.14  0.60          48.13          3.20         0.21
Ridge                           3.41  4.91  0.64          42.25          3.24         0.22
Random Forest                   3.67  5.76  0.50          48.13          3.64         0.37

Mejor modelo por validación cruzada: Ridge

Error medio (g) por grupo, modelo vs baseline:
                 error_abs  error_abs_baseline
grupo                                         
base                  4.06                4.01
plato_preparado       2.19                2.80
salsa                 2.20                1.34
snack_dulce           2.61                2.58

Error medio (g) Hacendado (1) vs otras marcas (0):
             error_abs       error_abs_baseline      
                  mean count      

## 4. Segunda ronda: la categoría hace el trabajo grueso, el modelo afina

**Lo que vi en la primera ronda.** Ridge y Random Forest no mejoraban al baseline. Ridge intenta aprender la proteína desde cero y no aprovecha bien que la categoría ya explica gran parte.

**La idea nueva.** Un modelo "baseline + corrección": parte de la media de la categoría y Ridge solo aprende cuánto se desvía un producto de esa media según su nombre y su precio. Pruebo 5 valores de la penalización (alpha) para esta variante y para Ridge solo.

**Cómo evito engañarme.** Como ya vi el test en la primera ronda, esta vez **elijo el modelo solo con validación cruzada** sobre el entrenamiento. El test se mira una única vez, para el modelo elegido y el baseline.

**Qué mirar en la salida:** la tabla de validación cruzada (menor MAE = mejor), la comparación final en test, el porcentaje de productos donde el modelo acierta más que el baseline y dónde están los errores grandes.

## 5. ¿Es real la mejora? Fugas y rango de error

La mejora de la segunda ronda es pequeña. Antes de darla por buena compruebo cuatro cosas:

1. **¿La mejora se distingue del azar?** Repito miles de veces el test remuestreando los productos (bootstrap) y miro el intervalo de la mejora media. Si incluye el 0, no puedo asegurar que el modelo sea mejor.
2. **¿Cuánto cambiaría el resultado sin el control de fugas?** Comparo la validación agrupada por variante con una mezclada. Si la mezclada sale mucho mejor, queda demostrado que sin el control los resultados serían falsamente buenos.
3. **¿Qué rango de error acompañará a cada estimación?** Con las predicciones "fuera de pliegue" del entrenamiento calculo el rango en el que cae el 80 % de los errores.
4. **¿En qué categorías es poco fiable?** El error medio por categoría me dirá dónde no conviene estimar.

In [4]:
from sklearn.base import BaseEstimator, RegressorMixin

class ResidualSobreBaseline(BaseEstimator, RegressorMixin):
    """Predice la media de la categoría + una corrección aprendida (Ridge) a partir del nombre y el precio."""
    def __init__(self, modelo=None):
        self.modelo = modelo
    def _base(self, X):
        return X["categoria_id"].map(self.media_cat_).fillna(self.media_global_).values
    def fit(self, X, y):
        y = pd.Series(np.asarray(y), index=X.index)
        self.media_cat_ = y.groupby(X["categoria_id"]).mean()
        self.media_global_ = y.mean()
        self.modelo.fit(X, y.values - self._base(X))
        return self
    def predict(self, X):
        return np.clip(self._base(X) + self.modelo.predict(X), 0, 100)

def ridge(alpha):
    return Pipeline([("pre", preprocesado()), ("m", Ridge(alpha=alpha))])

candidatos = {}
for a in [0.3, 1, 3, 10, 30]:
    candidatos[f"Ridge (alpha={a})"] = (lambda a=a: ridge(a))
    candidatos[f"Baseline + corrección Ridge (alpha={a})"] = (lambda a=a: ResidualSobreBaseline(ridge(a)))

# --- Elección SOLO con validación cruzada sobre entrenamiento ---
cv2 = {"Baseline (media por categoría)": []}
cv2.update({n: [] for n in candidatos})
for a_idx, b_idx in GroupKFold(n_splits=5).split(train, groups=train["variante"]):
    tr_f, va_f = train.iloc[a_idx], train.iloc[b_idx]
    cv2["Baseline (media por categoría)"].append(
        metricas(va_f[OBJETIVO].values, predecir_baseline(tr_f, va_f))["MAE"])
    for n, f in candidatos.items():
        m = f().fit(tr_f, tr_f[OBJETIVO])
        cv2[n].append(metricas(va_f[OBJETIVO].values, m.predict(va_f))["MAE"])

tabla_cv = pd.DataFrame({"MAE_cv_media": {n: np.mean(v) for n, v in cv2.items()},
                         "MAE_cv_desv": {n: np.std(v) for n, v in cv2.items()}}).round(2).sort_values("MAE_cv_media")
print("Validación cruzada (menor MAE = mejor):")
print(tabla_cv.to_string())

mejor_n = min(candidatos, key=lambda n: np.mean(cv2[n]))
print("\nModelo elegido por validación cruzada:", mejor_n)

# --- Evaluación final en test: una sola vez, solo el elegido y el baseline ---
modelo_final = candidatos[mejor_n]().fit(train, train[OBJETIVO])
t = test.copy()
t["pred"] = modelo_final.predict(test)
t["pred_baseline"] = predecir_baseline(train, test)
t["error_abs"] = (t["pred"] - t[OBJETIVO]).abs()
t["error_abs_baseline"] = (t["pred_baseline"] - t[OBJETIVO]).abs()

final = pd.DataFrame({
    "Baseline (media por categoría)": metricas(t[OBJETIVO].values, t["pred_baseline"].values),
    mejor_n: metricas(t[OBJETIVO].values, t["pred"].values)}).T.round(2)
print("\nTEST (datos no vistos):")
print(final.to_string())
print("\n% de productos donde el modelo se equivoca menos que el baseline:",
      round((t["error_abs"] < t["error_abs_baseline"]).mean() * 100, 1))

os.makedirs("resultados", exist_ok=True)
tabla_cv.to_csv("resultados/validacion_cruzada.csv")
final.to_csv("resultados/test_final.csv")

print("\nError medio (g) por grupo:")
print(t.groupby("grupo")[["error_abs", "error_abs_baseline"]].mean().round(2))
print("\nError medio (g) Hacendado (1) vs otras marcas (0):")
print(t.groupby("es_hacendado")[["error_abs", "error_abs_baseline"]].agg(["mean", "count"]).round(2))
print("\nCategorías con más error total en test:")
print(t.groupby("categoria_id")[["error_abs", "error_abs_baseline"]].agg(["mean", "count"]).round(2)
      .sort_values(("error_abs", "mean"), ascending=False).head(8))
print("\n10 mayores errores:")
print(t.sort_values("error_abs", ascending=False)[["nombre", "grupo", OBJETIVO, "pred"]].head(10).round(1).to_string())

Validación cruzada (menor MAE = mejor):
                                         MAE_cv_media  MAE_cv_desv
Baseline + corrección Ridge (alpha=3)            2.96         0.20
Baseline + corrección Ridge (alpha=10)           2.99         0.18
Baseline + corrección Ridge (alpha=1)            3.04         0.19
Baseline + corrección Ridge (alpha=30)           3.08         0.18
Baseline (media por categoría)                   3.20         0.21
Ridge (alpha=1)                                  3.24         0.22
Baseline + corrección Ridge (alpha=0.3)          3.25         0.12
Ridge (alpha=3)                                  3.29         0.26
Ridge (alpha=0.3)                                3.40         0.09
Ridge (alpha=10)                                 3.62         0.26
Ridge (alpha=30)                                 4.13         0.29

Modelo elegido por validación cruzada: Baseline + corrección Ridge (alpha=3)

TEST (datos no vistos):
                                        MAE  RMSE    

In [7]:
from sklearn.model_selection import KFold

nombres_cat = {"118": "Arroz", "121": "Legumbres", "120": "Pasta", "78": "Cereales", "80": "Galletas",
    "79": "Tortitas", "135": "Aceitunas y encurtidos", "133": "Frutos secos", "132": "Patatas fritas y snacks",
    "92": "Chocolate", "90": "Mermelada y miel", "48": "Aves y jamón cocido", "52": "Bacón y salchichas",
    "49": "Chopped y mortadela", "51": "Embutido curado", "50": "Jamón serrano", "58": "Paté y sobrasada",
    "54": "Queso curado", "56": "Queso lonchas y rallado", "53": "Queso untable y fresco", "43": "Embutido",
    "44": "Hamburguesas", "122": "Atún y conservas pescado", "123": "Berberechos y mejillones",
    "127": "Conservas verdura y fruta", "126": "Tomate", "103": "Yogures desnatados", "109": "Yogures griegos",
    "104": "Yogures naturales", "110": "Flan y natillas", "111": "Gelatina y postres", "77": "Huevos",
    "75": "Mantequilla y margarina", "60": "Pan de molde", "62": "Pan tostado y rallado",
    "64": "Picos y rosquilletas", "66": "Bollería envasada", "65": "Bollería de horno",
    "116": "Mayonesa, ketchup y mostaza", "117": "Otras salsas", "138": "Pizzas",
    "140": "Platos calientes", "142": "Platos fríos"}

# 1) ¿La mejora sobre el baseline es distinguible del azar? (bootstrap sobre el test)
rng = np.random.default_rng(42)
dif = (t["error_abs_baseline"] - t["error_abs"]).values     # > 0 = el modelo acierta más
boot = np.array([rng.choice(dif, size=len(dif), replace=True).mean() for _ in range(5000)])
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Mejora media del modelo sobre el baseline: {dif.mean():.2f} g (intervalo 95 %: {lo:.2f} a {hi:.2f})")
print("  -> si el intervalo incluye el 0, la mejora no se puede distinguir del azar")

# 2) ¿Cuánto cambiaría el resultado SIN el control de fugas? (mezclando variantes entre train y test)
def mae_cv(splitter, **kw):
    r, b = [], []
    for a_i, b_i in splitter.split(train, **kw):
        tr_f, va_f = train.iloc[a_i], train.iloc[b_i]
        m = candidatos[mejor_n]().fit(tr_f, tr_f[OBJETIVO])
        r.append(np.abs(m.predict(va_f) - va_f[OBJETIVO]).mean())
        b.append(np.abs(predecir_baseline(tr_f, va_f) - va_f[OBJETIVO]).mean())
    return np.mean(r), np.mean(b)
agr = mae_cv(GroupKFold(n_splits=5), groups=train["variante"])
mez = mae_cv(KFold(n_splits=5, shuffle=True, random_state=42))
print("\nMAE en validación cruzada (modelo / baseline):")
print(f"  Agrupada por variante (la que usamos): {agr[0]:.2f} / {agr[1]:.2f}")
print(f"  Mezclada (con fuga de variantes):      {mez[0]:.2f} / {mez[1]:.2f}")

# 3) Rango de error para el producto: residuos de predicciones "fuera de pliegue" en entrenamiento
oof = pd.Series(index=train.index, dtype=float)
for a_i, b_i in GroupKFold(n_splits=5).split(train, groups=train["variante"]):
    tr_f, va_f = train.iloc[a_i], train.iloc[b_i]
    oof.iloc[b_i] = candidatos[mejor_n]().fit(tr_f, tr_f[OBJETIVO]).predict(va_f)
residuo = train[OBJETIVO] - oof
q10, q90 = residuo.quantile([0.1, 0.9])
dentro = ((t[OBJETIVO] >= t["pred"] + q10) & (t[OBJETIVO] <= t["pred"] + q90)).mean() * 100
print(f"\nRango de error (80 %): estimación {q10:+.1f} g a {q90:+.1f} g")
print(f"Cobertura real en test (debería rondar el 80 %): {dentro:.1f} %")

# 4) ¿En qué categorías es poco fiable la estimación?
fiab = (train.assign(err=residuo.abs()).groupby("categoria_id")["err"]
        .agg(["mean", "count"]).query("count >= 8").sort_values("mean", ascending=False).round(1))
fiab.insert(0, "categoria", [nombres_cat.get(c, c) for c in fiab.index])
print("\nCategorías con peor error (validación cruzada, mínimo 8 productos):")
print(fiab.head(10).to_string())
print("\nCategorías con mejor error:")
print(fiab.tail(5).to_string())

Mejora media del modelo sobre el baseline: 0.03 g (intervalo 95 %: -0.18 a 0.24)
  -> si el intervalo incluye el 0, la mejora no se puede distinguir del azar

MAE en validación cruzada (modelo / baseline):
  Agrupada por variante (la que usamos): 2.96 / 3.20
  Mezclada (con fuga de variantes):      2.78 / 3.10

Rango de error (80 %): estimación -4.7 g a +3.6 g
Cobertura real en test (debería rondar el 80 %): 74.9 %

Categorías con peor error (validación cruzada, mínimo 8 productos):
                             categoria  mean  count
categoria_id                                       
132            Patatas fritas y snacks   9.1     29
133                       Frutos secos   6.5     35
121                          Legumbres   5.5     13
117                       Otras salsas   4.7     27
79                            Tortitas   4.4      8
56             Queso lonchas y rallado   4.2     18
122           Atún y conservas pescado   4.0     21
53              Queso untable y fresco   3.9

## 6. Modelo elegido y salida para el producto

**Decisión.** Como la mejora del modelo con ML no se distingue del azar, me quedo con **el baseline: la media de proteína de la categoría**. Es más simple, se explica en una frase y no depende del nombre del producto, así que es menos probable que falle con productos nuevos.

**Cómo lo hago fiable:**
- Una categoría es **fiable** si tiene al menos 8 productos y su error en validación cruzada es de 4 g o menos. En el resto no se estima. Las categorías con menos de 8 productos en el entrenamiento tampoco se estiman.
- El **rango de error del 80 %** se calcula solo con las categorías fiables.
- Compruebo en test, que no se usó para decidir nada, que el rango cubre alrededor del 80 % de los productos.

**Aplicación real.** Aplico la regla a los productos que no tienen datos nutricionales fiables y guardo la salida en `data/gold/estimaciones_proteina.csv`. Ese archivo es lo que consumirá el frontal: la proteína estimada, el rango y si la estimación es fiable. No se publica en el repositorio. Los resultados agregados (fiabilidad por categoría y cobertura de las estimaciones) sí se guardan en `resultados/`.

In [6]:
# 1) Errores fuera de pliegue del baseline en entrenamiento
oof_b = pd.Series(index=train.index, dtype=float)
for a_i, b_i in GroupKFold(n_splits=5).split(train, groups=train["variante"]):
    tr_f, va_f = train.iloc[a_i], train.iloc[b_i]
    oof_b.iloc[b_i] = predecir_baseline(tr_f, va_f)
res_b = train[OBJETIVO] - oof_b

# 2) Fiabilidad por categoría: al menos 8 productos y error medio <= 4 g
err_cat = res_b.abs().groupby(train["categoria_id"]).agg(["mean", "count"])
fiables = err_cat[(err_cat["count"] >= 8) & (err_cat["mean"] <= 4)].index
print("Categorías fiables:", len(fiables), "de", len(err_cat),
      "con productos en el entrenamiento (hay", len(nombres_cat), "en total)")
print("Categorías sin productos en el entrenamiento:",
      [nombres_cat[c] for c in nombres_cat if c not in err_cat.index])
print("Categorías NO fiables:", [nombres_cat.get(c, c) for c in err_cat.index.difference(fiables)])

# 3) Rango de error del 80 % usando solo categorías fiables
q10, q90 = res_b[train["categoria_id"].isin(fiables)].quantile([0.1, 0.9])
print(f"\nRango de error (80 %): {q10:+.1f} g a {q90:+.1f} g")

# 4) Comprobación en test (datos no vistos)
pred_b = predecir_baseline(train, test)
mask_te = test["categoria_id"].isin(fiables)
real = test[OBJETIVO]
dentro = ((real >= pred_b + q10) & (real <= pred_b + q90))[mask_te].mean() * 100
mae_fiables = np.abs(pred_b - real.values)[mask_te.values].mean()
mae_no_fiables = np.abs(pred_b - real.values)[~mask_te.values].mean()
print(f"Test, categorías fiables ({mask_te.sum()} productos): MAE {mae_fiables:.2f} g, "
      f"cobertura del rango {dentro:.1f} %")
print(f"Test, categorías no fiables ({(~mask_te).sum()} productos): MAE {mae_no_fiables:.2f} g")

# 5) Aplicación a los productos reales sin datos nutricionales fiables
silver_amp = pd.read_csv("data/silver/productos_silver_ampliado.csv", dtype={"ean": str})
sin = silver_amp[silver_amp["estado_calidad"] == "sin_datos"].copy()
sin["categoria_id"] = sin["categoria_id"].astype(str)
media_final = datos.groupby("categoria_id")[OBJETIVO].mean()

sin["proteina_estimada"] = sin["categoria_id"].map(media_final).round(1)
sin["fiable"] = sin["categoria_id"].isin(fiables)
sin["rango_bajo"] = (sin["proteina_estimada"] + q10).clip(lower=0).round(1)
sin["rango_alto"] = (sin["proteina_estimada"] + q90).round(1)
sin.loc[~sin["fiable"], ["proteina_estimada", "rango_bajo", "rango_alto"]] = np.nan

print("\nProductos sin datos nutricionales:", len(sin))
print("Con estimación fiable:", int(sin["fiable"].sum()), "| sin estimación:", int((~sin["fiable"]).sum()))
cobertura_est = sin.groupby("grupo")["fiable"].agg(con_estimacion="sum", total="count")
print(cobertura_est)

# Salida para el producto (no se publica: tiene nombres de productos)
salida = sin[["product_id", "ean", "nombre", "categoria_id", "grupo",
              "proteina_estimada", "rango_bajo", "rango_alto", "fiable"]]
salida.to_csv("data/gold/estimaciones_proteina.csv", index=False)

# Resultados agregados (sí se publican: solo cifras, sin datos de productos)
tabla_fiab = err_cat.round(2).rename(columns={"mean": "error_medio_g", "count": "productos"})
tabla_fiab["categoria"] = [nombres_cat.get(c, c) for c in tabla_fiab.index]
tabla_fiab["fiable"] = tabla_fiab.index.isin(fiables)
tabla_fiab.sort_values("error_medio_g").to_csv("resultados/fiabilidad_por_categoria.csv")
cobertura_est.to_csv("resultados/cobertura_estimaciones.csv")

print("\nEjemplos:")
print(salida[salida["fiable"]].head(8).to_string())

Categorías fiables: 26 de 42 con productos en el entrenamiento (hay 43 en total)
Categorías sin productos en el entrenamiento: ['Embutido']
Categorías NO fiables: ['Yogures naturales', 'Yogures griegos', 'Flan y natillas', 'Otras salsas', 'Legumbres', 'Patatas fritas y snacks', 'Frutos secos', 'Platos calientes', 'Platos fríos', 'Chopped y mortadela', 'Embutido curado', 'Queso curado', 'Queso lonchas y rallado', 'Bollería de horno', 'Huevos', 'Tortitas']

Rango de error (80 %): -3.4 g a +3.4 g
Test, categorías fiables (115 productos): MAE 2.08 g, cobertura del rango 82.6 %
Test, categorías no fiables (72 productos): MAE 5.48 g

Productos sin datos nutricionales: 269
Con estimación fiable: 120 | sin estimación: 149
                 con_estimacion  total
grupo                                 
base                         80    176
plato_preparado               2     18
salsa                         0      1
snack_dulce                  38     74

Ejemplos:
    product_id            ean  

## Conclusión

- **El ML no mejora de forma demostrable a la media por categoría.** La mejor variante baja el error medio en 0,03 g en test, con un intervalo del 95 % de −0,18 a +0,24 g, que incluye el cero. Ridge y Random Forest probados solos tampoco ganan al baseline en MAE.
- **La categoría explica casi todo.** El baseline ya tiene un R² de 0,60 en test.
- **Los errores grandes se concentran en categorías muy heterogéneas:** frutos secos (9,6 g de error medio en test) y patatas fritas y snacks (9,1 g en validación cruzada). Un mismo grupo mezcla productos con 1 g y con 50 g de proteína.
- **El control de fugas importa, aunque el efecto es pequeño:** sin él, el error baja de 2,96 g a 2,78 g, que sería un resultado demasiado optimista.
- **Para el producto:** estimo solo en las categorías fiables (26 de las 42 que tienen productos en el entrenamiento; la categoría Embutido no tiene ninguno). En test, el error medio en ellas es de 2,08 g y el rango del 80 % es de ±3,4 g, que lo cumple el 82,6 % de los productos. Eso cubre 120 de los 269 productos sin datos; los otros 149 necesitan otra fuente.
- **Limitaciones:** el test es pequeño (187 productos), el `estado` (seco o cocinado) no se usa en arroz, legumbres y pasta, el precio por litro se trata como precio por kilo, y los valores de partida vienen de Open Food Facts, que puede tener errores sin detectar.

La explicación completa está en `docs/entregas/03_machine_learning.md`.